# Lab: Cross-Platform Automation & System Execution Safety

### Instructions
Welcome to this week's lab! You will practice safely executing system commands using Python's `subprocess` module, handling platform differences cleanly, and avoiding security hazards like shell injection.

* **Execution Environments:** All exercises are designed to work across **macOS, Linux, and Windows** without modification.
* **Submission:** Complete all code cells, short answer text blocks, and the bonus challenge directly in this notebook file (`lab_subprocess_platform.ipynb`).

## Part 1: Short Answer Questions

### Question 1.1: `shell=True` vs. List Arguments
Explain why passing a command as a list (e.g., `["ping", "-c", "1", host]`) is safe against shell injection, whereas using `shell=True` with a string (e.g., `f"ping -c 1 {host}"`) creates a severe vulnerability.

*Passing arguments as a list bypasses the shell and sends arguments straight to the OS kernel, preventing special characters (;, |, &&) from being interpreted as extra commands. shell=True passes the full string to sh or cmd.exe, allowing attackers to chain and run arbitrary commands.*


### Question 1.2: Cross-Platform `os.chmod` 
Why is relying on `os.chmod()` problematic for enforcing security policies in a cross-platform Python tool that runs on both Linux and Windows?

*Windows does not support POSIX permission bits (read/write/execute for user/group/others). On Windows, os.chmod() only toggles the read-only flag, making it insufficient for enforcing detailed security policies across operating systems.*


## Part 2: Code Analysis & Bug Hunting

Read the following code snippets and answer the corresponding questions.

### Question 2.1: Analyzing Shell Injection Mechanics
Look at the following function:

```python
import subprocess

def run_diagnostics(target_ip: str):
    # Sends a single ping packet to target_ip
    cmd = f"ping -c 1 {target_ip}"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return result.stdout
```

1. **Vulnerability Identification:** What specific payload string could an attacker pass into `target_ip` to execute `whoami` (or `id`) alongside the ping command on Linux/macOS?
2. **Behavior Analysis:** What does `shell=True` actually instruct the operating system to do under the hood that allows this injection to happen?

*Type your answer here:*

1. Malicious Payload: 127.0.0.1; whoami
2. Explanation of OS behavior: shell=True hands the entire string to a system shell. The shell sees the ; (or &&) operator, completes the ping, and immediately executes whoami.


### Question 2.2: Environment Leaks & Scope
Analyze the code below:

```python
import os
import subprocess

def run_external_backup():
    # Set temporary token for backup process
    os.environ["BACKUP_AUTH_TOKEN"] = "secret_token_12345"
    subprocess.run(["backup_tool", "--run"])
    
    # ... rest of the application runs ...
```

1. What is the subtle risk with setting environment variables using `os.environ` directly in Python scripts?
2. How should this code be refactored using `subprocess.run` arguments to isolate the environment variable solely to the child process?

*Type your answer here:*

1. Risk: os.environ modifies global environment variables for the entire Python process and any future child processes, risking token leaks.
2. Refactoring approach: Scope the variable directly to the child process using the env parameter in subprocess.run(): subprocess.run(["backup_tool", "--run"], env={**os.environ, "BACKUP_AUTH_TOKEN": "secret_token_12345"})


## Part 3: Code Completion Exercises

Complete the missing sections marked with `# TODO` in the code blocks below.

### Exercise 3.1: Platform-Agnostic Executable Verification
Complete the function `check_dependencies` so that it uses `shutil.which` to verify that required CLI binaries are installed and accessible on the current system before running.

In [1]:
import shutil

def check_dependencies(tools: list[str]) -> dict[str, str | None]:
    tool_map = {}
    for tool in tools:
        tool_map[tool] = shutil.which(tool)
    return tool_map

# Test
test_tools = ["python", "python3", "git", "fake_tool_xyz_123"]
results = check_dependencies(test_tools)

for tool, path in results.items():
    status = f"Found at {path}" if path else "NOT FOUND"
    print(f"[{tool}]: {status}")

[python]: Found at C:\Users\monfe\Documents\sat4311-labs\Classroom\.venv\Scripts\python.EXE
[python3]: Found at C:\Users\monfe\AppData\Local\Microsoft\WindowsApps\python3.EXE
[git]: Found at C:\Program Files\Git\ucrt64\bin\git.EXE
[fake_tool_xyz_123]: NOT FOUND


### Exercise 3.2: Safe Execution with Timeouts and Error Checking
Complete the function `safe_execute` to execute system commands cleanly. 

Requirements:
1. Arguments must be passed as a list.
2. Must capture text output.
3. Must enforce a strict timeout (`timeout_sec`).
4. Must catch `subprocess.TimeoutExpired` and return a descriptive error string without crashing.

In [2]:
import subprocess

def safe_execute(cmd_args: list[str], timeout_sec: int = 5) -> str:
    try:
        result = subprocess.run(
            cmd_args, 
            capture_output=True, 
            text=True, 
            timeout=timeout_sec
        )
        if result.returncode == 0:
            return result.stdout.strip()
        else:
            return f"Error (code {result.returncode}): {result.stderr.strip()}"
            
    except subprocess.TimeoutExpired:
        return f"Error: Command '{' '.join(cmd_args)}' timed out after {timeout_sec}s."

# Test
import sys
output = safe_execute([sys.executable, "--version"], timeout_sec=2)
print("Output:", output)

Output: Python 3.14.8


### Exercise 3.3: Cross-Platform Adapter with `platform.system()`
Complete the class/function dispatcher to select the correct directory listing command based on the operating system.

In [3]:
import platform
import subprocess

def get_platform_ls_command() -> list[str]:
    system_os = platform.system()
    match system_os:
        case "Windows":
            return ["cmd.exe", "/c", "dir"]
        case "Linux" | "Darwin":
            return ["ls", "-l"]
        case _:
            raise RuntimeError(f"Unsupported OS: {system_os}")

# Test
cmd = get_platform_ls_command()
print(f"Detected command for {platform.system()}: {cmd}")
res = subprocess.run(cmd, capture_output=True, text=True)
print("\nFirst 3 lines of output:")
print("\n".join(res.stdout.splitlines()[:3]))

Detected command for Windows: ['cmd.exe', '/c', 'dir']

First 3 lines of output:
 Volume in drive C is OS
 Volume Serial Number is CE19-CB92



## Part 4: Implement a Cross-Platform System Hostname & Architecture Collector
Write a function `get_system_info()` that gathers basic host info by executing CLI binaries safely.

**Requirements:**
1. Do **not** use `shell=True` under any circumstances.
2. Retrieve the host's **hostname** using the system command `hostname` across all platforms.
3. Retrieve the host's **OS release/kernel version** by conditionally choosing the tool:
   * **Windows:** Execute `["cmd.exe", "/c", "ver"]` 
   * **Linux / macOS:** Execute `["uname", "-r"]` 
4. Use `shutil.which` to verify the commands exist before executing.
5. Ensure a timeout of 3 seconds per command execution.
6. Return a dictionary structured as:
   ```python
   {
       "platform": "...",      # from platform.system()
       "hostname": "...",      # from stdout of 'hostname'
       "os_kernel": "..."      # from stdout of 'ver' or 'uname -r'
   }
#### **Important Notes:** 
1. &lt;variable&gt;.stdout.strip() will be helpful for acquiring correct output.
2. Everything you need exists in your previous solutions and the previous codeblocks.
   ```

In [4]:
import platform
import shutil
import subprocess

def get_system_info() -> dict[str, str]:
    current_os = platform.system()
    info = {
        "platform": current_os,
        "hostname": "Unknown",
        "os_kernel": "Unknown"
    }

    # EXCEPTION HANDLING
    try:
        if shutil.which("hostname"):
            res = subprocess.run(["hostname"], capture_output=True, text=True, timeout=3)
            if res.returncode == 0:
                info["hostname"] = res.stdout.strip()

        kernel_cmd = ["cmd.exe", "/c", "ver"] if current_os == "Windows" else ["uname", "-r"]

        if shutil.which(kernel_cmd[0]):
            res = subprocess.run(kernel_cmd, capture_output=True, text=True, timeout=3)
            if res.returncode == 0:
                info["os_kernel"] = res.stdout.strip()

    except subprocess.TimeoutExpired:  # EXCEPTION HANDLING
        print("Command timed out.")
    except Exception as e:  # EXCEPTION HANDLING
        print(f"Error occurred: {e}")

    return info

# Test
print("System Inventory Output:")
print(get_system_info())

System Inventory Output:
{'platform': 'Windows', 'hostname': 'Emmetts_ROG', 'os_kernel': 'Microsoft Windows [Version 10.0.26300.9457]'}


### BONUS: exceptions handling
1. Add exception handling to your code and mark it with # EXCEPTION HANDLING
2. subprocess.run() has a returncode attribute that you can call.